# Single-task augmentation generalization overfitting on Modal

This experiment asks whether one fresh full FLUX transformer can learn the shared rule of ARC task `0d3d703e` from 512 augmented training views and transfer it to 128 held-out views of the same task. The pool uses four effective geometry classes (`ID`, `R90`, `R180`, and `R270`); redundant reflections are excluded. NVARC and the FLUX VAE remain frozen.

Every training view receives exactly eight contacts: one shared-noise endpoint update at `t=1` and seven stratified updates. The expensive target-free evaluation is intentionally limited to fixed panels: 32 seen plus 32 held-out views at steps 0, 128, and 256, then 128 seen plus all 128 held-out views at step 512. The Original view remains a separate diagnostic.

## 1. Immutable runtime configuration

Before running, set `DIFFUSION_ARC_REVISION` to the exact pushed 40-character commit containing this notebook and its runner. Branch names, abbreviated hashes, and missing revisions are rejected. Automatic shutdown is enabled by default; set `DIFFUSION_ARC_AUTO_SHUTDOWN=0` only when the container must remain available for debugging.

In [ ]:
from __future__ import annotations

import base64
from collections import deque
from datetime import datetime, timezone
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time

VOLUME_NAME = 'diffusion-arc-models'
VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = "6beb492787a5c2c031c8cc2004caa6a7412517f6"
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'
PROTOCOL = 'augmentation-generalization-overfit'
RESULT_FAMILY = 'single-task-augmentation-generalization-overfitting'
EXPECTED_STAGE = 'augmentation-generalization-overfit'
EXPECTED_FINAL_STEP = 512
EXPECTED_EVALUATION_STEPS = [0, 128, 256, 512]
PREVIEW_PAGE_LIMIT = 12
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-augmentation-generalization-source-{RUN_ID}'
REPORT_PATH = VOLUME_ROOT / 'results' / RESULT_FAMILY / RUN_ID / 'report.json'

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}

def require_exact_commit(revision: str) -> None:
    if len(revision) != 40 or any(
        character not in '0123456789abcdefABCDEF' for character in revision
    ):
        raise ValueError(
            'DIFFUSION_ARC_REVISION must be an exact pushed 40-character commit'
        )

def run_command(
    label: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-50:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()

def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def should_stream_runner_line(line: str) -> bool:
    normalized = line.strip()
    if not normalized:
        return False
    lowered = normalized.lower()
    return (
        normalized.startswith('[')
        or normalized.startswith(f'{EXPECTED_STAGE}:')
        or f'{EXPECTED_STAGE} step ' in normalized
        or normalized.startswith('FINAL_REPORT=')
        or any(
            marker in lowered
            for marker in ('traceback', 'error', 'failed', 'failure', 'non-finite')
        )
    )

def report_protocol_name(report: dict) -> str | None:
    protocol = report.get('protocol')
    if isinstance(protocol, dict):
        return protocol.get('name')
    return protocol if isinstance(protocol, str) else None

def expected_exposure_counts(audit: dict) -> dict:
    expected = {}
    for key in ('expected_counts', 'expected_per_view', 'expected'):
        candidate = audit.get(key)
        if isinstance(candidate, dict):
            expected = candidate
            break
    return {
        'total': expected.get('total'),
        'endpoint': expected.get('endpoint', expected.get('shared_endpoint')),
        'stratified': expected.get('stratified', expected.get('regular')),
    }

def final_comparison_pages(report: dict) -> list[Path]:
    evaluations = report['stages'][EXPECTED_STAGE].get('evaluations', [])
    if not evaluations:
        return []
    candidates: list[str] = []

    def collect(value, key_hint: str = '') -> None:
        if isinstance(value, dict):
            for key, child in value.items():
                collect(child, key.lower())
        elif isinstance(value, list):
            for child in value:
                collect(child, key_hint)
        elif isinstance(value, str) and (
            'sheet' in key_hint or 'page' in key_hint
        ):
            candidates.append(value)

    collect(evaluations[-1])
    pages: list[Path] = []
    seen: set[Path] = set()
    for value in candidates:
        candidate = Path(value)
        if not candidate.is_absolute():
            candidate = REPORT_PATH.parent / candidate
        candidate = candidate.resolve()
        if candidate in seen or not candidate.is_file():
            continue
        seen.add(candidate)
        pages.append(candidate)
    return pages

def schedule_modal_shutdown() -> None:
    if not AUTO_SHUTDOWN:
        print('Automatic shutdown disabled by DIFFUSION_ARC_AUTO_SHUTDOWN=0.')
        return
    modal_task_id = os.environ.get('MODAL_TASK_ID', '').strip()
    if not modal_task_id:
        print('WARNING: MODAL_TASK_ID is unavailable; stop the container manually.')
        return
    shutdown_code = (
        'import subprocess, sys, time; time.sleep(15); '
        "subprocess.run(['modal', 'container', 'stop', '--yes', sys.argv[1]], check=False)"
    )
    subprocess.Popen(
        [sys.executable, '-c', shutdown_code, modal_task_id],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    print('Modal shutdown scheduled in 15 seconds.')

print('Run ID:', RUN_ID)
print('Protocol:', PROTOCOL)
print('Requested source revision:', SOURCE_REVISION or '<missing>')
print('Final report:', REPORT_PATH)


## 2. Fetch, verify, train, summarize, and shut down

This single protected cell fetches a detached source commit, installs the pinned runtime only when required, runs the complete unit-test suite, launches the augmentation-generalization protocol, verifies pool and exposure contracts, verifies all four evaluations plus the final checkpoint and report upload, previews paginated final comparison sheets, and schedules shutdown in a `finally` block after success or failure. Per-update progress bars are suppressed from notebook output; only milestones and failures are streamed.

In [ ]:
REPORT = None
run_succeeded = False
runner_tail: deque[str] = deque(maxlen=80)

try:
    require_exact_commit(SOURCE_REVISION)
    if not VOLUME_ROOT.is_dir():
        raise FileNotFoundError(f'Modal Volume mount is missing: {VOLUME_ROOT}')
    if REPORT_PATH.exists():
        raise FileExistsError(REPORT_PATH)

    gpu_summary = run_command(
        'query GPU',
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    )
    print('GPU:', gpu_summary)

    SOURCE_DIR.mkdir(parents=True)
    run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
    run_command(
        'add origin',
        ['git', 'remote', 'add', 'origin', REPOSITORY_URL],
        cwd=SOURCE_DIR,
    )

    github_token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not github_token:
        raise RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
    credential = base64.b64encode(
        f'x-access-token:{github_token}'.encode('utf-8')
    ).decode('ascii')
    git_environment = os.environ.copy()
    git_environment.pop('GITHUB_TOKEN', None)
    git_environment.update({
        'GIT_TERMINAL_PROMPT': '0',
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
    })
    try:
        run_command(
            'fetch source',
            ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
            cwd=SOURCE_DIR,
            env=git_environment,
        )
        run_command(
            'checkout source',
            ['git', 'checkout', '--detach', 'FETCH_HEAD'],
            cwd=SOURCE_DIR,
        )
    finally:
        git_environment.clear()
        del credential

    resolved_source_commit = run_command(
        'resolve source',
        ['git', 'rev-parse', 'HEAD'],
        cwd=SOURCE_DIR,
    )
    if resolved_source_commit.lower() != SOURCE_REVISION.lower():
        raise RuntimeError('Fetched source commit differs from requested revision')

    run_command(
        'remove incompatible torchvision',
        [sys.executable, '-m', 'pip', 'uninstall', '--yes', 'torchvision'],
    )
    requirements = [
        f'{name}=={version}'
        for name, version in PINNED_PACKAGES.items()
        if installed_version(name) != version
    ]
    if requirements:
        run_command(
            'install pinned dependencies',
            [
                sys.executable, '-m', 'pip', 'install', '--quiet',
                '--no-input', '--no-cache-dir', *requirements,
            ],
        )
    run_command(
        'install Diffusion-ARC source',
        [
            sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
            '--no-deps', '--no-build-isolation', '-e', str(SOURCE_DIR),
        ],
    )

    source_path = str((SOURCE_DIR / 'src').resolve())
    test_environment = os.environ.copy()
    test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
    test_environment['PYTHONPATH'] = source_path
    test_started = time.perf_counter()
    test_process = subprocess.run(
        [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
        cwd=SOURCE_DIR,
        env=test_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if test_process.returncode != 0:
        print('\n'.join(test_process.stdout.splitlines()[-50:]))
        raise RuntimeError(f'Unit tests returned {test_process.returncode}')
    print(
        f'Resolved source {resolved_source_commit}; unit tests PASS in '
        f'{time.perf_counter() - test_started:.1f}s.'
    )

    runner = (
        SOURCE_DIR / 'scripts' / 'experiments'
        / 'run_single_task_augmentation_generalization.py'
    )
    if not runner.is_file():
        raise RuntimeError(
            'Selected source revision lacks the augmentation-generalization runner'
        )
    runner_command = [
        sys.executable, '-u', str(runner), '--protocol', PROTOCOL,
        '--volume-root', str(VOLUME_ROOT), '--volume-name', VOLUME_NAME,
        '--run-id', RUN_ID, '--source-commit', resolved_source_commit,
    ]
    runner_environment = os.environ.copy()
    runner_environment.update({
        'PYTHONDONTWRITEBYTECODE': '1',
        'PYTHONPATH': source_path,
        'HF_HUB_OFFLINE': '1',
        'TRANSFORMERS_OFFLINE': '1',
        'HF_HUB_DISABLE_TELEMETRY': '1',
        'HF_HUB_DISABLE_PROGRESS_BARS': '1',
        'TOKENIZERS_PARALLELISM': 'false',
    })
    process = subprocess.Popen(
        runner_command,
        cwd=SOURCE_DIR,
        env=runner_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for raw_line in process.stdout:
        for fragment in raw_line.replace('\r', '\n').splitlines():
            normalized = fragment.strip()
            if not normalized:
                continue
            runner_tail.append(normalized)
            if should_stream_runner_line(normalized):
                print(normalized)
    return_code = process.wait()
    if return_code != 0:
        if REPORT_PATH.is_file():
            failed_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
            print(json.dumps({
                'status': failed_report.get('status'),
                'failed_stage': failed_report.get('failed_stage'),
                'failures': failed_report.get('failures'),
            }, indent=2))
        raise RuntimeError(
            f'Augmentation-generalization runner exited with {return_code}'
        )

    REPORT = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
    if REPORT.get('status') != 'COMPLETE':
        raise RuntimeError(f'Experiment report is not COMPLETE: {REPORT.get("status")}')
    if report_protocol_name(REPORT) != PROTOCOL:
        raise RuntimeError('Report protocol differs from the requested protocol')
    pool_counts = REPORT.get('pool', {}).get('counts')
    expected_pool_counts = {'train': 512, 'holdout': 128, 'original': 1}
    if pool_counts != expected_pool_counts:
        raise RuntimeError(
            f'Pool counts differ: {pool_counts} versus {expected_pool_counts}'
        )
    if set(REPORT.get('stages', {})) != {EXPECTED_STAGE}:
        raise RuntimeError('Report contains an unexpected stage set')
    stage = REPORT['stages'][EXPECTED_STAGE]
    if stage.get('status') != 'COMPLETE':
        raise RuntimeError(f'{EXPECTED_STAGE} is not COMPLETE')

    training = stage.get('training', {})
    flow_sampling = training.get('flow_sampling', {})
    expected_updates = {'shared_endpoint': 64, 'stratified': 448}
    if flow_sampling.get('update_counts') != expected_updates:
        raise RuntimeError(
            f'Update counts differ: {flow_sampling.get("update_counts")} '
            f'versus {expected_updates}'
        )
    expected_exposures = {
        'shared_endpoint': 512, 'stratified': 3584, 'total': 4096,
    }
    if flow_sampling.get('exposure_counts') != expected_exposures:
        raise RuntimeError(
            f'Exposure counts differ: {flow_sampling.get("exposure_counts")} '
            f'versus {expected_exposures}'
        )
    exposure_audit = training.get('per_view_exposure_audit', {})
    if exposure_audit.get('all_exact') is not True:
        raise RuntimeError('Not every training view received its exact exposure budget')
    expected_per_view = {'total': 8, 'endpoint': 1, 'stratified': 7}
    if expected_exposure_counts(exposure_audit) != expected_per_view:
        raise RuntimeError(
            'Per-view exposure contract differs: '
            f'{expected_exposure_counts(exposure_audit)} versus {expected_per_view}'
        )

    evaluations = stage.get('evaluations', [])
    evaluation_steps = [entry.get('step') for entry in evaluations]
    if evaluation_steps != EXPECTED_EVALUATION_STEPS:
        raise RuntimeError(
            f'Evaluation steps differ: {evaluation_steps} versus '
            f'{EXPECTED_EVALUATION_STEPS}'
        )
    final_panels = evaluations[-1].get('panels', {})
    expected_final_panels = {'seen': 128, 'holdout': 128, 'original': 1}
    measured_final_panels = {
        name: final_panels.get(name, {}).get('count')
        for name in expected_final_panels
    }
    if measured_final_panels != expected_final_panels:
        raise RuntimeError(
            f'Final panel counts differ: {measured_final_panels} versus '
            f'{expected_final_panels}'
        )

    checkpoint = stage.get('saved_checkpoint')
    if not isinstance(checkpoint, dict):
        raise RuntimeError('Final transformer checkpoint is missing')
    if checkpoint.get('step') != EXPECTED_FINAL_STEP:
        raise RuntimeError('Final transformer checkpoint step is not 512')
    if not checkpoint.get('remote_file_sizes_verified'):
        raise RuntimeError('Final transformer checkpoint is not remotely verified')
    durability = REPORT.get('durability', {})
    if not durability.get('remote_content_verified'):
        raise RuntimeError('Final report content was not verified after upload')

    summary = {
        'run_id': RUN_ID,
        'source_commit': resolved_source_commit,
        'report': str(REPORT_PATH),
        'remote_report': durability.get('remote_report_path'),
        'stage': EXPECTED_STAGE,
        'pool_counts': pool_counts,
        'update_counts': flow_sampling.get('update_counts'),
        'exposure_counts': flow_sampling.get('exposure_counts'),
        'evaluation_steps': evaluation_steps,
        'final_panel_counts': measured_final_panels,
        'checkpoint': checkpoint.get('path'),
        'checkpoint_step': checkpoint.get('step'),
    }
    print(json.dumps(summary, indent=2))

    preview_paths = final_comparison_pages(REPORT)
    if not preview_paths:
        print('Final paginated GT-vs-Pred comparison sheets are not present.')
    else:
        try:
            from IPython.display import Image as DisplayImage, display

            print(
                f'Final comparison pages: {len(preview_paths)}; previewing '
                f'{min(len(preview_paths), PREVIEW_PAGE_LIMIT)}.'
            )
            for page_number, preview_path in enumerate(
                preview_paths[:PREVIEW_PAGE_LIMIT], start=1
            ):
                print(f'Page {page_number}: {preview_path}')
                display(DisplayImage(filename=str(preview_path), width=1100))
            if len(preview_paths) > PREVIEW_PAGE_LIMIT:
                print(
                    f'{len(preview_paths) - PREVIEW_PAGE_LIMIT} additional pages '
                    'remain available beside the report.'
                )
        except Exception as preview_error:
            print(f'Could not preview final comparison pages: {preview_error}')
    run_succeeded = True
finally:
    if not run_succeeded and runner_tail:
        print('Runner output tail:')
        print('\n'.join(tuple(runner_tail)[-20:]))
    schedule_modal_shutdown()


## Interpretation guardrails

- Seen exact-grid results measure fit to the augmented training pool; held-out exact-grid results measure transfer to new augmentations of the same ARC task.
- The Original view is reported separately and is not silently included in either denominator.
- The fixed 128-view seen panel estimates training fit; it is not a complete evaluation of all 512 training views.
- The shared `t=1` schedule is held constant from the successful endpoint run, so this experiment primarily changes augmentation diversity and per-view repetition.
- `COMPLETE` means that artifacts and protocol contracts were verified; it does not imply that the learning or generalization criterion passed.